In [ ]:
import os
import pandas as pd
import numpy as np

# -----------------------
# Config / constants
# -----------------------
MAPPING_PATH = "b1a_mapping.csv"

TOTAL_PRIVATE  = "CES0500000001"
GOVERNMENT     = "CES9000000001"
TOTAL_NONFARM  = "CES0000000001"


WIDE_PQ = "b1a_wide_seriesid.parquet"
WIDE_CSV = "b1a_wide_seriesid.csv"

# -----------------------
# Helpers
# -----------------------
def norm_industry_code(x: str) -> str:
    if pd.isna(x):
        return ""
    s = "".join(ch for ch in str(x) if ch.isdigit())
    return s.zfill(8)

def load_wide():
    if os.path.exists(WIDE_PQ):
        return pd.read_parquet(WIDE_PQ)
    if os.path.exists(WIDE_CSV):
        df = pd.read_csv(WIDE_CSV, index_col=0, parse_dates=True)
        if not isinstance(df.index, pd.DatetimeIndex):
            df.index = pd.to_datetime(df.index, errors="coerce")
        return df
    if "wide" in globals():
        return globals()["wide"]
    raise FileNotFoundError("Could not find wide panel (parquet/csv) and no in-memory `wide` found.")

# ---- load mapping ----
m = pd.read_csv(MAPPING_PATH, dtype=str)
m.columns = m.columns.str.strip()
m = m.applymap(lambda v: v.strip() if isinstance(v, str) else v)

m["industry_code_8"] = m["industry_code"].apply(norm_industry_code)

# industry_code -> series_id lookup
code_to_series = (
    m.dropna(subset=["industry_code_8", "series_id"])
     .drop_duplicates(subset=["industry_code_8"])
     .set_index("industry_code_8")["series_id"]
     .to_dict()
)

def compute_parent(ind8: str, sid: str):
    """
    Final rules:

    1) TOTAL_NONFARM -> itself
    2) TOTAL_PRIVATE -> TOTAL_NONFARM
    3) GOVERNMENT -> TOTAL_NONFARM
    4) Any other government series (industry_code starts with '90')
       -> GOVERNMENT
    5) Private:
         - if digits 6-8 == '000' -> TOTAL_PRIVATE
         - else:
             * try 3-digit parent: first5 + '000'
             * try 4-digit parent: first6 + '00'
                 - if equals itself or missing -> try 2-digit parent: first4 + '0000'
             * if 2-digit missing -> TOTAL_PRIVATE
    """

    if sid == TOTAL_NONFARM:
        return TOTAL_NONFARM, "self_total_nonfarm"

    if sid == TOTAL_PRIVATE:
        return TOTAL_NONFARM, "total_private_to_total_nonfarm"

    if sid == GOVERNMENT:
        return TOTAL_NONFARM, "government_to_total_nonfarm"

    if not ind8 or len(ind8) < 8:
        return TOTAL_PRIVATE, "fallback_total_private"

    first2 = ind8[:2]
    suffix_6_8 = ind8[5:]

    # Government subseries
    if first2 == "90":
        return GOVERNMENT, "gov_subseries_to_government"

    # Top-level private
    if suffix_6_8 == "000":
        return TOTAL_PRIVATE, "top_level_to_total_private"

    # ---- detailed private ----

    # 3-digit parent
    parent3_code = ind8[:5] + "000"
    parent3_sid = code_to_series.get(parent3_code)
    if parent3_sid and parent3_sid != sid:
        return parent3_sid, "parent_3digit"

    # 4-digit parent
    parent4_code = ind8[:6] + "00"
    parent4_sid = code_to_series.get(parent4_code)
    if parent4_sid and parent4_sid != sid:
        return parent4_sid, "parent_4digit"

    # 2-digit parent  (UPDATED CORRECT RULE)
    parent2_code = ind8[:4] + "0000"
    parent2_sid = code_to_series.get(parent2_code)
    if parent2_sid and parent2_sid != sid:
        return parent2_sid, "parent_2digit"

    return TOTAL_PRIVATE, "fallback_total_private"

parents = m.apply(lambda r: compute_parent(r["industry_code_8"], r["series_id"]), axis=1)
m["parent_series_id"] = parents.apply(lambda t: t[0])
m["parent_rule"] = parents.apply(lambda t: t[1])

OUT_MAPPING = "b1a_mapping_with_parent.csv"
m.to_csv(OUT_MAPPING, index=False)

/var/folders/x3/1282rh0s7_b240x5mjx1zlmc0000gn/T/ipykernel_95978/271943798.py:42: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  m = m.applymap(lambda v: v.strip() if isinstance(v, str) else v)


In [9]:
# ---- diagnostics (compact but clear) ----
parents_per_series = m.groupby("series_id")["parent_series_id"].nunique(dropna=True)
num_series_multiple_parents = int((parents_per_series > 1).sum())

detailed_mask = (m["industry_code_8"].str[5:] != "000").fillna(False) & ~m["industry_code_8"].str.startswith("90", na=False)
fallback_detailed_mask = detailed_mask & m["parent_rule"].eq("fallback_total_private")
num_detailed_fallbacks = int(fallback_detailed_mask.sum())

print("DIAGNOSTICS")
print("----------")
print("Rows:", len(m))
print("Unique series:", m["series_id"].nunique(dropna=True))
print("Unique parents used:", m["parent_series_id"].nunique(dropna=True))
print("Series with >1 parent assigned:", num_series_multiple_parents)
print("Detailed private series that fell back to TOTAL_PRIVATE:", num_detailed_fallbacks)
print("Parent rule counts:\n", m["parent_rule"].value_counts().to_string())

DIAGNOSTICS
----------
Rows: 842
Unique series: 842
Unique parents used: 87
Series with >1 parent assigned: 0
Detailed private series that fell back to TOTAL_PRIVATE: 1
Parent rule counts:
 parent_rule
parent_3digit                     638
top_level_to_total_private        104
parent_4digit                      56
parent_2digit                      20
gov_subseries_to_government        20
self_total_nonfarm                  1
total_private_to_total_nonfarm      1
fallback_total_private              1
government_to_total_nonfarm         1


In [10]:
# ---- compute shares ----
wide = load_wide()

m2 = m[m["series_id"].isin(wide.columns)].copy()
shares = pd.DataFrame(index=wide.index)

for sid, denom in zip(m2["series_id"], m2["parent_series_id"]):
    if denom not in wide.columns:
        shares[sid] = np.nan
    else:
        shares[sid] = wide[sid] / wide[denom]

OUT_SHARES = "b1a_employment_shares.csv"
shares.to_csv(OUT_SHARES)

print(f"\nSaved updated mapping: {OUT_MAPPING}")
print(f"Saved shares: {OUT_SHARES}")

/var/folders/x3/1282rh0s7_b240x5mjx1zlmc0000gn/T/ipykernel_95978/3504762393.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  shares[sid] = wide[sid] / wide[denom]
/var/folders/x3/1282rh0s7_b240x5mjx1zlmc0000gn/T/ipykernel_95978/3504762393.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  shares[sid] = wide[sid] / wide[denom]
/var/folders/x3/1282rh0s7_b240x5mjx1zlmc0000gn/T/ipykernel_95978/3504762393.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times,


Saved updated mapping: b1a_mapping_with_parent.csv
Saved shares: b1a_employment_shares.csv
